In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install -q sentence-transformers scikit-learn pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 88.1 MB/s eta 0:00:00:00:0100:01


In [3]:
# 📦 Cell 1: Imports
import random
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

# 📦 Cell 2: Generate iPhone Product Map
def generate_iphone_product_map():
    product_map = {}
    for model in range(8, 21):
        base = f"iPhone {model}"
        product_map[base] = [
            base, f"IP {model}", f"I Phone {model}", f"Apple phone {model}",
            f"IPHONE {model}", f"i-phone {model}", f"Iphone{model}", f"ip{model}"
        ]
        if model >= 11:
            pro = f"iPhone {model} Pro"
            product_map[pro] = [
                pro, f"IP {model} Pro", f"I Phone {model} Pro", f"Apple phone {model} Pro",
                f"IPHONE {model} PRO", f"i-phone {model} Pro", f"Iphone{model}Pro", f"ip{model}pro"
            ]
            prom = f"iPhone {model} Pro Max"
            product_map[prom] = [
                prom, f"IP {model} Pro Max", f"I Phone {model} Pro Max", f"Apple phone {model} Pro Max",
                f"IPHONE {model} PRO MAX", f"i-phone {model} Pro Max", f"Iphone{model}ProMax", f"ip{model}promax"
            ]
        if model >= 14:
            plus = f"iPhone {model} Plus"
            product_map[plus] = [
                plus, f"IP {model} Plus", f"I Phone {model} Plus", f"Apple phone {model} Plus",
                f"IPHONE {model} PLUS", f"i-phone {model} Plus", f"Iphone{model}Plus", f"ip{model}plus"
            ]
        if model in [8, 13, 16]:
            se = f"iPhone SE ({model})"
            product_map[se] = [
                se, f"iPhone SE {model}", f"SE {model}", f"Apple SE {model}",
                f"iPhone Special Edition {model}", f"IPHONE SE {model}", f"iphone se {model}"
            ]
    return product_map

# 📦 Cell 3: Additional Apple Products
def get_extended_apple_product_map():
    iphone_map = generate_iphone_product_map()
    macbook_gens = ["M1", "M2", "M3", "M4"]
    macbook_tiers = ["", "Pro", "Max", "Ultra"]
    macbook_sizes = [("Air", ["13", "15"]), ("Pro", ["14", "16"])]

    additional_products = {
        # iPads
        "iPad Air (5th generation)": ["iPad Air 5", "iPadA5", "Air 5", "iPad Gen 5", "iPad Air fifth gen"],
        "iPad Pro 12.9-inch (6th generation)": ["iPad Pro 12.9", "Pro 12.9", "Apple iPad Pro 6", "iPP 12.9", "iPad Pro 6th"],
        "iPad Mini (6th generation)": ["iPad Mini 6", "Mini 6", "iPM6", "Apple iPad Mini 6", "iPad Mini sixth gen"],
        "iPad (10th generation)": ["iPad 10", "iPad Gen 10", "Apple iPad 10", "iPad tenth gen", "iPad 10th"],

        # Apple Watch
        "Apple Watch Ultra 2": ["Apple Watch Ultra 2", "AWU2", "Watch Ultra 2", "Apple Ultra 2", "Ultra 2"],
        "Apple Watch Series 9": ["Apple Watch 9", "AW 9", "Watch Series 9", "Series 9", "S9 Watch"],
        "Apple Watch SE (2nd generation)": ["Apple Watch SE 2", "AW SE 2", "Watch SE 2", "Apple SE 2", "SE Watch 2"],

        # AirPods
        "AirPods Pro (2nd generation)": ["AirPods Pro 2", "Air Pods Pro 2", "APP2", "Pro Buds 2", "Apple Pro Buds"],
        "AirPods Max": ["AirPods Max", "AP Max", "Air Pods Max", "Apple headphones max"]
    }

    for chip in macbook_gens:
        for tier in macbook_tiers:
            chip_name = f"{chip} {tier}".strip()
            for model, sizes in macbook_sizes:
                for size in sizes:
                    name = f"MacBook {model} {size}-inch ({chip_name})"
                    aliases = [
                        f"MacBook {model} {size} {chip_name}", f"{model[:1]}B{model[1:]} {size} {chip_name}", f"Mac {model} {size} {chip_name}",
                        f"Apple laptop {size} {chip_name}", f"Macbook{model}{size} {chip_name}", f"{model} {size} {chip_name}"
                    ]
                    additional_products[name] = list(set(aliases))

    return {**iphone_map, **additional_products}

# 📦 Cell 4: Processor Map
def generate_apple_processor_map():
    generations = ["M1", "M2", "M3", "M4"]
    tiers = ["", "Pro", "Max", "Ultra"]
    processor_map = {}
    for gen in generations:
        for tier in tiers:
            full_name = f"{gen} {tier}".strip()
            canonical = full_name
            aliases = list(set([
                canonical, canonical.upper(), canonical.lower(), canonical.replace(" ", ""),
                f"Apple {canonical}", f"{canonical} chip", f"{canonical} processor", f"{canonical} silicon"
            ]))
            processor_map[canonical] = aliases
    return processor_map

# 📦 Cell 5: Synthetic Comment Generator
def generate_synthetic_comments(product_map, n_per_product=100):
    templates = [
        "Just bought the {} — loving it so far!",
        "Thinking about upgrading to the {}.",
        "Anyone else having issues with their {}?",
        "I prefer my {} over other brands.",
        "The {} is so sleek and fast!",
        "Can't believe how good the {} is!",
        "Got the {} yesterday. Wow.",
        "My old {} just broke 😭",
        "The battery on my {} drains fast.",
        "Is the {} worth the upgrade?"
    ]
    data = []
    for product, aliases in product_map.items():
        for _ in range(n_per_product):
            alias = random.choice(aliases)
            template = random.choice(templates)
            comment = template.format(alias)
            data.append((comment, product))
    return pd.DataFrame(data, columns=["text", "label"])

# 📦 Cell 6: Combine, Embed, Train
product_map = get_extended_apple_product_map()
processor_map = generate_apple_processor_map()
df_products = generate_synthetic_comments(product_map, n_per_product=50)
df_processors = generate_synthetic_comments(processor_map, n_per_product=30)
df_combined = pd.concat([df_products, df_processors], ignore_index=True)

# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_combined["label"])

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")
X = model.encode(df_combined["text"].tolist(), show_progress_bar=True)

# Train classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X, y)

# Inference function
def identify_apple_product(comment):
    embedding = model.encode([comment])
    pred = clf.predict(embedding)
    return label_encoder.inverse_transform(pred)[0]

# Basic Examples
examples = [
    "Can't wait to try the iPhone 20 Pro Max!",
    "Got a new MBP 16 M4 Ultra yesterday.",
    "How good is the M3 Max in the Mac Air 13?",
    "Airpods Pro 2 are amazing",
    "iPad 10 is great for drawing"
]

for text in examples:
    print(f"> {text} -> {identify_apple_product(text)}")


2025-07-28 12:26:03.579028: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753705563.911122      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753705564.005503      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (13 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

> Airpods Pro 2 are amazing -> AirPods Pro (2nd generation)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

> iPad 10 is great for drawing -> iPad (10th generation)


In [4]:
# Test Examples
additional_examples = [
    "Just upgraded to the IPHONE 14 Plus and it's a beast!",
    "Mac Air 15 M3 Pro handles video editing really well.",
    "I’m loving my new Apple Watch Ultra 2!",
    "Can’t decide between iPad Mini 6 and iPad 10th gen.",
    "Battery life on the Air Pods Max is so good.",
    "The M4 Ultra chip is blazing fast in my new MacBook Pro.",
    "Is the iPhone SE 16 any good compared to IP 16?",
    "Bought the Apple laptop 14 M2 Max and it flies!",
    "Series 9 Watch is much smoother than my old S6.",
    "I still think the MBA 13 M1 Max is the best deal."
]

print("\n# Additional Test Predictions:")
for example in additional_examples:
    print(f"> {example} -> {identify_apple_product(example)}")



# 🔍 Additional Test Predictions:


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

> Just upgraded to the IPHONE 14 Plus and it's a beast! -> iPhone 14 Plus


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

> Mac Air 15 M3 Pro handles video editing really well. -> MacBook Air 15-inch (M3 Pro)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (11 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

> Series 9 Watch is much smoother than my old S6. -> Apple Watch Series 9


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

> I still think the MBA 13 M1 Max is the best deal. -> MacBook Air 13-inch (M1 Max)


In [5]:
#Edge Examples 
# 📦 Cell 8: Edge Case Test Examples
edge_case_examples = [
    "iphn 16 pro is 🔥🔥🔥",  # typo for iPhone
    "Got the MBP14M4Ultra 💻 and it's INSANE!",
    "watch s9 is slick af 😎",
    "airpd max noise canceling >>>>",
    "ipad prooo 12.9 = beast for sketching",
    "macbookair13 m2max is best value rn",
    "thinking of upgrding to apple phne 15",  # typo
    "just copped the ip16pro… worth it??",
    "mb air 15 ultra chip is zooming ⚡",
    "m3pro in pro 14 inch macbook = 🚀"
]

print("\n# 🧪 Edge Case Predictions:")
for example in edge_case_examples:
    print(f"> {example} -> {identify_apple_product(example)}")



# 🧪 Edge Case Predictions:


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

> iphn 16 pro is 🔥🔥🔥 -> iPhone 16 Pro


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

> Got the MBP14M4Ultra 💻 and it's INSANE! -> MacBook Pro 14-inch (M4)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (11 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

> mb air 15 ultra chip is zooming ⚡ -> MacBook Air 15-inch (M2 Ultra)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

> m3pro in pro 14 inch macbook = 🚀 -> MacBook Pro 14-inch (M3 Pro)
